In [11]:
import numpy as np

import mw_analysis.utils as utils
import mw_analysis.snirf as snirf

CHANNELS = [
    "rx1_l1", "rx1_l2", "rx1_l3", "rx1_l4",
    "rx1_l5", "rx1_l6", "rx1_l7", "rx1_l8",
    "rx1_l9", "rx1_l10", "rx1_l11", "rx1_l12",
    "rx1_l13", "rx1_l14", "rx1_l15", "rx1_l16",
    "rx2_l1", "rx2_l2", "rx2_l3", "rx2_l4",
    "rx2_l5", "rx2_l6", "rx2_l7", "rx2_l8",
    "rx2_l9", "rx2_l10", "rx2_l11", "rx2_l12",
    "rx2_l13", "rx2_l14", "rx2_l15", "rx2_l16",
    "battery_voltage", "timestamp"
]

MAPPINGS = {
    "S1_D1": ("rx1_l1", "rx1_l2"),
    "S1_D2": ("rx1_l3", "rx1_l4"),
    "S1_D3": ("rx1_l5", "rx1_l6"),
    "S1_D4": ("rx1_l7", "rx1_l8"),
    "S2_D1": ("rx2_l9", "rx2_l10"),
    "S2_D2": ("rx2_l11", "rx2_l12"),
    "S2_D3": ("rx2_l13", "rx2_l14"),
    "S2_D4": ("rx2_l15", "rx2_l16"),
}

# Processed datasets
datasets = []

for subject_id in range(1, 9):
    stream_df, events_df = utils.mw_h5_to_df(f"../data/participant{subject_id}-artinis.h5", CHANNELS)

    # Clean the events df
    clean_df = utils.clean_events(events_df)

    # Convert to HbO/Hb
    hbo_df = stream_df[["timestamp"]]

    for name, (ch1, ch2) in MAPPINGS.items():

        if utils.sci(stream_df[ch1].values, stream_df[ch2].values) < 0.9:
            hbo_df[f"{name} hbo"] = np.full(len(stream_df), np.nan)
            hbo_df[f"{name} hb"] = np.full(len(stream_df), np.nan)

        else:
            HbO, Hb = utils.mbll(stream_df[ch1].values, stream_df[ch2].values)
            hbo_df[f"{name} hbo"] = utils.iir_filter(HbO)
            hbo_df[f"{name} hb"] = utils.iir_filter(Hb)

    # Add to the dataset
    dataset = snirf.NirscordDataset(subject_id, hbo_df, clean_df)

    datasets.append(dataset)

    # Write to SNIRF file
    snirf.to_snirf(dataset, f"../data/processed/participant{subject_id}-artinis.snirf")


In [12]:
import plotly.graph_objects as go


colours = ["red", "orange", "yellow", "green", "blue", "purple", "brown", "pink"]

for dataset in datasets:
    fig = go.Figure()

    #
    time = np.arange(dataset.stream_df.iloc[-1]["timestamp"])

    # --- left channel
    fig_left = go.Figure()

    for i, name in enumerate(MAPPINGS.keys()):
        fig_left.add_trace(go.Scatter(
            x=time, y=dataset.stream_df[f"{name} hbo"],
            mode='lines', name=f'{name} Δ[HbO]',
            line=dict(color=colours[i]),
            hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
        ))

    fig_left.update_layout(
        title=f'Participant {dataset.subject_id} HbO concentrations',
        xaxis_title='Time (s)',
        yaxis_title='Δ Concentration (μM)',
        hovermode='x unified'
    )
    fig_left.show()

# Convert to SNIRF

In [13]:
import itertools

error_epochs = []
no_error_epochs = []

def extract_epochs(stream_df, events_df):
    epochs = []

    for index, row in events_df.iterrows():
        epochs.append(stream_df[
            (stream_df["timestamp"] >= row.timestamp - 30) &
            (stream_df["timestamp"] < row.timestamp + 10)
        ].reset_index(drop=True))

    return epochs

for dataset in datasets:
    error_epochs.append(extract_epochs(dataset.stream_df, dataset.events_df[dataset.events_df["value"] == 0]))
    no_error_epochs.append(extract_epochs(dataset.stream_df, dataset.events_df[dataset.events_df["value"] == 1]))

error_epochs = list(itertools.chain(*error_epochs))
no_error_epochs = list(itertools.chain(*no_error_epochs))


# Convert to HbO and Hb Concentrations

In [14]:
import pandas as pd

# --- time axis in seconds
time = np.arange(-30, 10)



# ---


average_sart_errors = pd.concat(error_epochs, axis=0).groupby(level=0).mean().dropna()
average_sart_no_errors = pd.concat(no_error_epochs, axis=0).groupby(level=0).mean().dropna()

for i, name in enumerate(MAPPINGS.keys()):
    fig_left = go.Figure()
    fig_left.add_trace(go.Scatter(
        x=time, y=average_sart_errors[f"{name} hbo"],
        mode='lines', name='SART Error',
        line=dict(color='green'),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))
    fig_left.add_trace(go.Scatter(
        x=time, y=average_sart_no_errors[f"{name} hbo"],
        mode='lines', name='SART No Error',
        line=dict(color='blue'),
        hovertemplate='Time: %{x:.2f} s<br>Δ[HbO]: %{y:.4f} μM'
    ))
    fig_left.update_layout(
        title=f'Average SART {name} HbO concentrations',
        xaxis_title='Time (s)',
        yaxis_title='Δ Concentration (μM)',
        hovermode='x unified'
    )
    fig_left.show()

